In [0]:
import json
import base64
import time
import datetime
from google import genai
from youtube_transcript_api import YouTubeTranscriptApi
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

# Setup
GEMINI_API_KEY = dbutils.secrets.get(scope="courseify", key="gemini-api")
client = genai.Client(api_key=GEMINI_API_KEY)
COURSE_ID = "py_jenny"

# Mana Core 25 Topics Syllabus
SYLLABUS = [
    {"id": 0, "title": "Introduction & Setup"}, {"id": 1, "title": "Variables & Data Types"},
    {"id": 2, "title": "Type Casting"}, {"id": 3, "title": "Basic I/O"},
    {"id": 4, "title": "Operators"}, {"id": 5, "title": "Conditional Statements"},
    {"id": 6, "title": "Loops & Loop Control"}, {"id": 7, "title": "Core Data Structures (Lists, Tuples, Sets, Dictionaries)"},
    {"id": 8, "title": "Comprehensions"}, {"id": 9, "title": "Functions Basics & Arguments"},
    {"id": 10, "title": "Scope (Local vs Global)"}, {"id": 11, "title": "Lambda Functions"},
    {"id": 12, "title": "Built-in Functions"}, {"id": 13, "title": "Modules & Packages"},
    {"id": 14, "title": "Classes & Objects"}, {"id": 15, "title": "Attributes & Methods"},
    {"id": 16, "title": "Inheritance & Polymorphism"}, {"id": 17, "title": "Encapsulation & Abstraction"},
    {"id": 18, "title": "Magic/Dunder Methods"}, {"id": 19, "title": "File Handling"},
    {"id": 20, "title": "Exception Handling"}, {"id": 21, "title": "Iterators & Generators"},
    {"id": 22, "title": "Decorators"}, {"id": 23, "title": "Regular Expressions (Regex)"},
    {"id": 24, "title": "Advanced Concepts Overview"}
]

def get_transcript(video_id):
    try:
        transcript_list = YouTubeTranscriptApi.get_transcript(video_id)
        return " ".join([item['text'] for item in transcript_list])
    except Exception: return ""

def map_videos_to_topics(videos_list):
    print("🧠 AI Mapping videos to 25 main topics...")
    syllabus_str = "\n".join([f"ID {s['id']}: {s['title']}" for s in SYLLABUS])
    videos_str = "\n".join([f"VID {v['video_id']} : {v['video_title']}" for v in videos_list])
    
    prompt = f"""
    Map the following YouTube videos to the most relevant Syllabus Topic. Group similar videos under the same topic_id.
    SYLLABUS TOPICS:
    {syllabus_str}
    
    VIDEOS:
    {videos_str}
    
    Return EXACTLY a JSON array. Format: [{{"video_id": "...", "topic_id": 0, "refined_title": "Clean Name"}}]
    """
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash', 
            contents=prompt, 
            config={'temperature': 0.1, 'response_mime_type': 'application/json'}
        )
        return json.loads(response.text)
    except Exception as e:
        print(f"Mapping Error: {e}")
        return []

def generate_ai_content(combined_transcript, topic_title):
    prompt = f"""
    You are an Elite Coding Instructor. Generate a Topper-Level study guide and 5-10 multiple-choice questions for the Topic: '{topic_title}'.
    
    OUTPUT FORMAT (EXACTLY VALID JSON):
    {{
        "markdown_notes": "Detailed A-to-Z notes...",
        "practice_test": [
            {{"question": "?", "options": ["A", "B", "C", "D"], "answer_index": 0}}
        ]
    }}
    Source Transcript: {combined_transcript[:25000]}
    """
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash', 
            contents=prompt, 
            config={'temperature': 0.3, 'response_mime_type': 'application/json'}
        )
        return json.loads(response.text)
    except Exception as e:
        print(f"Content Gen Error: {e}")
        return None

def process_bronze_to_silver():
    # 1. Read Raw Data from Bronze
    bronze_df = spark.sql(f"SELECT video_id, video_title, channel_name FROM courseify.default.course_bronze WHERE course_id = '{COURSE_ID}'")
    videos = [row.asDict() for row in bronze_df.collect()]
    
    if not videos:
        print("No data in Bronze table yet!")
        return

    # 2. Get AI Mapping
    mapping = map_videos_to_topics(videos)
    
    # 3. Group by the 25 Main Topics
    course_structure = {s['id']: {"title": s['title'], "videos": []} for s in SYLLABUS}
    for m in mapping:
        topic_id = m.get('topic_id')
        if topic_id in course_structure:
            course_structure[topic_id]['videos'].append(m)

    silver_data = []
    current_time = datetime.datetime.now()

    # 4. Generate Content for each Main Topic
    for topic_id, data in course_structure.items():
        if not data['videos']: continue # Skip if no videos mapped
            
        print(f"\n⏳ Processing Silver Topic: {data['title']}")
        
        # Check if already processed in Silver table
        existing = spark.sql(f"SELECT 1 FROM courseify.default.course_silver WHERE course_id = '{COURSE_ID}' AND main_topic_id = {topic_id}").count()
        if existing > 0:
            print(f"⏩ Topic {data['title']} already in Silver DB. Skipping.")
            continue
            
        combined_transcript = ""
        for v in data['videos']:
            t = get_transcript(v['video_id'])
            if t: combined_transcript += f"\n{t}"
            
        if not combined_transcript:
            combined_transcript = f"Generate comprehensive notes for {data['title']}."

        ai_data = generate_ai_content(combined_transcript, data['title'])
        
        if ai_data:
            # 5. Extract Notes and Set Flags (1 or 0)
            raw_notes = ai_data.get('markdown_notes', "")
            quiz_data = json.dumps(ai_data.get('practice_test', []))
            
            notes_flag = 1 if raw_notes else 0
            qa_flag = 1 if ai_data.get('practice_test') else 0
            b64_notes = base64.b64encode(raw_notes.encode('utf-8')).decode('utf-8') if notes_flag else ""

            videos_json = json.dumps(data['videos'])
            
            silver_data.append((
                COURSE_ID, int(topic_id), data['title'], videos_json,
                notes_flag, qa_flag, b64_notes, quiz_data, current_time
            ))
            print(f"   ✅ Silver Data Ready! (Notes: {notes_flag}, QA: {qa_flag})")
            
        time.sleep(30) # Prevent API rate limits

    # 6. Save to Silver Table
    if silver_data:
        schema = StructType([
            StructField("course_id", StringType(), True),
            StructField("main_topic_id", IntegerType(), True),
            StructField("main_topic_name", StringType(), True),
            StructField("sub_videos_json", StringType(), True),
            StructField("notes_status", IntegerType(), True),
            StructField("qa_status", IntegerType(), True),
            StructField("notes_data", StringType(), True),
            StructField("qa_data", StringType(), True),
            StructField("updated_at", TimestampType(), True)
        ])
        df = spark.createDataFrame(silver_data, schema=schema)
        df.write.format("delta").mode("append").saveAsTable("courseify.default.course_silver")
        print("\n🎉 Job 2 Success! Data processed and saved to Silver Table.")
    else:
        print("\n✅ Silver Table is already up to date.")

if __name__ == "__main__":
    process_bronze_to_silver()